In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys
sys.path.append("/storage/groups/ml01/workspace/jan.domke/Code/")
import scib_excerpts


In [2]:
atlas_data = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/atlas_data.h5ad")

In [3]:
query_data = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/harmonized_ipf_data.h5ad")

### set labels to use for integration:

check which cells have only level 1 annotations:

In [4]:
lev1_obs = query_data.obs.loc[query_data.obs.ann_highest_res == 1, :].copy()
print("Number of cells with only level 1 annotation:", lev1_obs.shape[0])

Number of cells with only level 1 annotation: 62705


In [5]:
# count number of cells per ann_level_1 group that have no further annotations
lev1_annotations = lev1_obs.groupby("ann_level_1").agg({"ann_level_1":"count"}).rename(columns={"ann_level_1":"n_cells_w_only_l1_ann"})
# count total cells in each ann_level_1 group
lev1_annotations['total'] = query_data.obs.groupby("ann_level_1").agg({"ann_level_1":"count"}).loc[lev1_annotations.index,"ann_level_1"]

In [6]:
lev1_annotations

,n_cells_w_only_l1_ann,total
ann_level_1,,
Endothelial,5008,20633
Epithelial,3997,63910
Immune,963,357408
Stroma,52737,65508


In [7]:
np.unique(query_data.obs[query_data.obs.ann_level_1=="Stroma"].iloc[:,20:25].ann_level_2)

array(['1_Stroma', 'Fibroblast lineage', 'Mesothelium', 'Smooth muscle'],
      dtype=object)

Set all  cells as "unlabeled" (since they only make up a small part of their group)

In [8]:
lev1_labeled = []

In [9]:
lev1_unlabeled = query_data.obs.index[[max_res == 1 for max_res, lev1label in zip(query_data.obs.ann_highest_res, query_data.obs.ann_level_1)]]

In [10]:
len(lev1_labeled)

0

In [11]:
len(lev1_unlabeled)

62705

In [12]:
lev1_annotations.n_cells_w_only_l1_ann.sum() == len(lev1_labeled) + len(lev1_unlabeled)

True

Now check which cells have only up to level 2 annotations:

In [13]:
lev2_obs = query_data.obs.loc[query_data.obs.ann_highest_res == 2, :].copy()
print("Number of cells with only level 1/2 annotation:", lev2_obs.shape[0])

Number of cells with only level 1/2 annotation: 15892


In [14]:
# count number of cells per ann_level_2 group that have no further annotations
lev2_annotations = lev2_obs.groupby("ann_level_2").agg({"ann_level_2":"count"}).rename(columns={"ann_level_2":"n_cells_w_only_l2_ann"})
# count total cells in each ann_level_2 group
lev2_annotations['total'] = query_data.obs.groupby("ann_level_2").agg({"ann_level_2":"count"}).loc[lev2_annotations.index,"ann_level_2"]

In [15]:
lev2_annotations

,n_cells_w_only_l2_ann,total
ann_level_2,,
1_Endothelial,0,5008
1_Epithelial,0,3997
1_Immune,0,963
1_Stroma,0,52737
Airway epithelium,0,41522
Alveolar epithelium,0,18391
Blood vessels,7280,12621
Fibroblast lineage,2936,10099
Lymphatic EC,3004,3004


We will set all cells with level 2 annotations "Chondrocytes", "Fibroblast lineage", "Schwann cells", "Lymphatic EC", "Mesothelium", "Smooth muscle" to their level 2 annotations.

In [16]:
lev2_labels_to_keep = ["Fibroblast lineage", "Lymphatic EC", "Mesothelium", "Smooth muscle"]
lev2_labeled = query_data.obs.index[[lev2_lab in lev2_labels_to_keep for lev2_lab in query_data.obs.ann_level_2]]

In [17]:
len(lev2_labeled) == lev2_annotations.loc[lev2_labels_to_keep,"total"].sum()

True

In [18]:
lev2_unlabeled = query_data.obs.index[[max_res == 2 and lev2_lab not in lev2_labels_to_keep for max_res, lev2_lab in zip(query_data.obs.ann_highest_res, query_data.obs.ann_level_2)]]

In [19]:
len(lev2_unlabeled) == lev2_annotations.loc[~lev2_annotations.index.isin(lev2_labels_to_keep),"n_cells_w_only_l2_ann"].sum()

True

now generate scANVI labels:

In [20]:
query_data.obs['scanvi_label'] = query_data.obs.ann_level_3.tolist()
query_data.obs.loc[lev1_labeled,'scanvi_label'] = query_data.obs.loc[lev1_labeled, "ann_level_1"]
query_data.obs.loc[lev1_unlabeled,'scanvi_label'] = "unlabeled"
query_data.obs.loc[lev2_labeled,'scanvi_label'] = query_data.obs.loc[lev2_labeled, "ann_level_2"]
query_data.obs.loc[lev2_unlabeled,'scanvi_label'] = "unlabeled"

In [21]:
[x for x in query_data.obs['scanvi_label'].unique() if x not in atlas_data.obs["scanvi_label"].unique()]

['Erythroblast']

save dataset 

In [22]:
query_data.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/IPF_labeled.h5ad")

... storing 'scanvi_label' as categorical


In [ ]:
#query_data = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/IPF_labeled.h5ad")

### Normalize counts

In [23]:
#Save raw counts
query_data.layers["counts"] = query_data.X.copy()

In [24]:
query_data.obs.dataset.unique()

['Kaminski_2020', 'Banovich_Kropski_2020_disease', 'Misharin_Budinger_2018_disease', 'Sheppard_2020']
Categories (4, object): ['Kaminski_2020', 'Banovich_Kropski_2020_disease', 'Misharin_Budinger_2018_disease', 'Sheppard_2020']

In [25]:
adams = query_data[(query_data.obs.dataset=="Kaminski_2020"),:]
haberman = query_data[(query_data.obs.dataset=="Banovich_Kropski_2020_disease"),:]
reyfman = query_data[(query_data.obs.dataset=="Misharin_Budinger_2018_disease"),:]
tsukui = query_data[(query_data.obs.dataset=="Sheppard_2020"),:]


/opt/python/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)


In [ ]:
# normalize 
adams_norm = scib_excerpts.SCRAN_normalize(adams, log_transform=False)

R[write to console]: Loading required package: SingleCellExperiment

R[write to console]: Loading required package: SummarizedExperiment

R[write to console]: Loading required package: MatrixGenerics

R[write to console]: Loading required package: matrixStats

R[write to console]: 
Attaching package: ‘MatrixGenerics’


R[write to console]: The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowC

In [27]:
# store
adams_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/adams_norm.h5ad")

In [28]:
# normalize 
haberman_norm = scib_excerpts.SCRAN_normalize(haberman, log_transform=False)

/opt/python/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)
R[write to console]: dimnames(.) <- NULL:  translated to 
dimnames(.) <- list(NULL,NULL)  <==>  unname(.)



In [29]:
# store
haberman_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/haberman_norm.h5ad")

In [30]:
# normalize 
reyfman_norm = scib_excerpts.SCRAN_normalize(reyfman, log_transform=False)

/opt/python/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)
R[write to console]: dimnames(.) <- NULL:  translated to 
dimnames(.) <- list(NULL,NULL)  <==>  unname(.)



In [ ]:
# store
reyfman_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/reyfman_norm.h5ad")

In [ ]:
# normalize 
tsukui_norm = scib_excerpts.SCRAN_normalize(tsukui, log_transform=False)

/opt/python/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)
R[write to console]: dimnames(.) <- NULL:  translated to 
dimnames(.) <- list(NULL,NULL)  <==>  unname(.)



In [ ]:
# store
tsukui_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/tsukui_norm.h5ad")

In [ ]:
final_tmp1 = adams_norm.concatenate(haberman_norm)
final_tmp2 = final_tmp1.concatenate(reyfman_norm)
final = final_tmp2.concatenate(tsukui_norm)

In [ ]:
# store
final.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/IPF_labeled_normalized.h5ad")

... storing 'dataset' as categorical
... storing 'study' as categorical
... storing 'original_celltype_ann' as categorical
... storing 'condition' as categorical
... storing 'subject_ID' as categorical
... storing 'sample' as categorical
... storing 'single_cell_platform' as categorical
... storing 'sample_type' as categorical
... storing 'ann_level_2' as categorical
... storing 'ann_level_3' as categorical
... storing 'ann_level_4' as categorical
... storing 'ann_level_5' as categorical
... storing 'ann_level_2_clean' as categorical
... storing 'ann_level_3_clean' as categorical
... storing 'ann_level_4_clean' as categorical
... storing 'ann_level_5_clean' as categorical
... storing 'scanvi_label' as categorical


In [ ]:
final = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/IPF_labeled_normalized.h5ad")